# Liberías, carga y exploración inicial

Primero, importamos las librerías necesarias y cargamos los datos.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Carga de datos
df = pd.read_csv('../data/users_behavior.csv')

# Inspección rápida
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


# Segmentación de datos

Dado que no se tiene un conjunto de prueba separado, se propone dividir los datos en tres partes:
- Entrenamiento (60%)
- Validación(20%)
- Prueba o test (20%)

In [2]:
# Separar características (features) y objetivo (target)
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# Primera división: 60% entrenamiento, 40% para (validación + prueba)
df_train_features, df_temp_features, df_train_target, df_temp_target = train_test_split(
    features, target, test_size=0.40, random_state=12345)

# Segunda división: Dividir el 40% restante a la mitad (20% validación, 20% prueba)
df_valid_features, df_test_features, df_valid_target, df_test_target = train_test_split(
    df_temp_features, df_temp_target, test_size=0.50, random_state=12345)

# Investigación de Modelos

Se prueba tres algoritmos diferentes ajustando sus hiperparámetros para encontrar el que supere el umbral de 0.75

- Árbol de decisión
    - se itera sobre la profundidad máxima (max_depth) para evitar el sobreajuste.
- Bosque aleatorio
    - Se ajusta el número de estimadores (n_estimators)
- Regresión Logistica

In [3]:
best_depth = 0
best_accuracy_dt = 0

for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(df_train_features, df_train_target)
    predictions = model.predict(df_valid_features)
    acc = accuracy_score(df_valid_target, predictions)
    if acc > best_accuracy_dt:
        best_accuracy_dt = acc
        best_depth = depth

print(f"Mejor Árbol de Decisión: Exactitud = {best_accuracy_dt:.4f} en profundidad {best_depth}")

Mejor Árbol de Decisión: Exactitud = 0.7854 en profundidad 3


In [4]:
best_est = 0
best_accuracy_rf = 0

for est in range(10, 101, 10):
    model = RandomForestClassifier(random_state=12345, n_estimators=est)
    model.fit(df_train_features, df_train_target)
    acc = model.score(df_valid_features, df_valid_target)
    if acc > best_accuracy_rf:
        best_accuracy_rf = acc
        best_est = est

print(f"Mejor Random Forest: Exactitud = {best_accuracy_rf:.4f} con {best_est} estimadores")

Mejor Random Forest: Exactitud = 0.7916 con 50 estimadores


In [5]:
model_lr = LogisticRegression(random_state=12345, solver='liblinear')
model_lr.fit(df_train_features, df_train_target)
acc_lr = model_lr.score(df_valid_features, df_valid_target)
print(f"Exactitud Regresión Logística: {acc_lr:.4f}")

Exactitud Regresión Logística: 0.7589


Como se puede ver el algoritmo con el cuál se logró la mejor exactitud, fue con Random Forest, de 0.7916 con 50 estimadores 

# Evaluación final con conjunto de prueba
Sabiendo que tuvimos la mejor exactitud con Random Forest, ahora evaluamos su calidad con los datos de prueba que el modelo nunca ha visto. 

In [6]:
final_model = RandomForestClassifier(random_state=12345, n_estimators=best_est)
final_model.fit(df_train_features, df_train_target)
test_acc = final_model.score(df_test_features, df_test_target)

print(f"Exactitud final en el conjunto de prueba: {test_acc:.4f}")

Exactitud final en el conjunto de prueba: 0.7932


# Prueba de cordura
Para que un modelo sea útil, debe ser mejor que el azar o que una estrategia simplista (como predecir siempre la clase más frecuente).

En nuestro dataset, la mayoría de los usuarios están en el plan Smart (0). Si un modelo predijera "0" para todos, ¿qué exactitud tendría?

In [7]:
# Verificamos la proporción de la clase mayoritaria
sanity_check = target.value_counts(normalize=True)
print("Frecuencia de clases:\n", sanity_check)

Frecuencia de clases:
 is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64


Hallazgo: Si el plan Smart representa el aproximadamente el 70% de los datos, cualquier modelo con una exactitud inferior a 0.70 es "peor" que simplemente adivinar siempre el plan más común. El modelo debe superar este valor base significativamente. 

Se logro superar el umbral de exactitud propuesto del 0.75, con el 0.7932 de exactitud. 

# Resumen de Hallazgos

- Decision Tree: Es rápido pero propenso a sobreajustar si la profundidad es muy alta.
- Random Forest: Suele ofrecer la mayor exactitud al promediar múltiples árboles, superando fácilmente el 0.75.
- Logistic Regression: Es el más estable y rápido, pero a veces demasiado simple para patrones de comportamiento complejos.